# Hostile Testing Phase 3 - Data, Hygiene, Development Modules

## Purpose
Expand testing coverage by testing data, hygiene, and development modules with correct imports.

## Goal
Add ~12 more functions to reach 3%+ coverage

In [1]:
import sys
import pandas as pd
import tempfile
from pathlib import Path
import shutil
import json

test_results = {
    'function': [],
    'module': [],
    'test_category': [],
    'passed': [],
    'error_message': [],
    'severity': []
}

def record_test(function, module, test_category, passed, error_message="", severity="medium"):
    test_results['function'].append(function)
    test_results['module'].append(module)
    test_results['test_category'].append(test_category)
    test_results['passed'].append(passed)
    test_results['error_message'].append(error_message)
    test_results['severity'].append(severity)

def hostile_test(func, test_name, *args, **kwargs):
    try:
        result = func(*args, **kwargs)
        return (True, result, "")
    except Exception as e:
        return (False, None, str(e))

print("Test harness loaded for Phase 3")

Test harness loaded for Phase 3


## Section 1: Data Module - Sample Data Functions

In [2]:
from siege_utilities import list_available_datasets, get_dataset_info

# Test list_available_datasets
success, result, error = hostile_test(list_available_datasets, "list_datasets")
record_test("list_available_datasets", "data.sample_data", "basic_call", success, error, "low")
print(f"list_available_datasets: {'✅' if success else '❌'}")

# Test get_dataset_info with hostile names
hostile_names = [
    None, "", "../../../etc/passwd", "'; DROP TABLE datasets; --",
    "<script>alert('xss')</script>", "A" * 10000, "dataset\x00name",
    "../../sensitive", "nonexistent"
]

for name in hostile_names:
    success, result, error = hostile_test(get_dataset_info, "hostile_dataset", name)
    record_test("get_dataset_info", "data.sample_data", "hostile_names", success, error,
                "high" if "../" in str(name) else "medium")

print(f"Completed {len([r for r in test_results['module'] if r == 'data.sample_data'])} data tests")

Importing logging from siege_utilities.core.logging
Successfully imported 15 functions from logging
Importing string_utils from siege_utilities.core.string_utils
Successfully imported 1 functions from string_utils
siege_utilities.core: Imported 16 functions


[siege_utilities] 2025-10-13 08:48:02,789 INFO: Importing hdfs_operations from siege_utilities.distributed.hdfs_operations


[siege_utilities] 2025-10-13 08:48:02,789 INFO: Successfully imported 2 functions from hdfs_operations


[siege_utilities] 2025-10-13 08:48:02,789 INFO: Importing hdfs_config from siege_utilities.distributed.hdfs_config


[siege_utilities] 2025-10-13 08:48:02,791 INFO: Successfully imported 5 functions from hdfs_config


...

[siege_utilities] 2025-10-13 08:48:03,095 INFO: DuckDB not available - using standard geospatial stack


[siege_utilities] 2025-10-13 08:48:03,264 WARNING: Could not import additional analytics utilities: cannot import name 'get_dataset_metadata' from 'siege_utilities.analytics.datadotworld_connector' (/Users/dheerajchand/Documents/Professional/Siege_Analytics/Code/siege_utilities/siege_utilities/analytics/datadotworld_connector.py)


[siege_utilities] 2025-10-13 08:48:03,264 WARNING: Could not import reporting utilities: No module named 'siege_utilities.reporting.base_template'


[siege_utilities] 2025-10-13 08:48:03,265 WARNING: Could not import chart generation functions: No module named 'siege_utilities.reporting.base_template'


INFO: Importing environment from siege_utilities.testing.environment
INFO: Successfully imported 9 functions from environment
INFO: Importing runner from siege_utilities.testing.runner
INFO: Successfully imported 6 functions from runner
list_available_datasets: ✅
Completed 10 data tests


## Section 2: Hygiene Module - Docstring Generation

In [3]:
from siege_utilities import (
    categorize_function, generate_docstring_template,
    find_python_files, process_python_file, analyze_function_signature
)

# Test categorize_function
hostile_funcs = [
    None, "", "'; DROP TABLE functions; --", "<script>alert(1)</script>",
    "../../../etc/passwd", "A" * 10000, "func\x00name", "get_", "create_", "数据库函数"
]

for fname in hostile_funcs:
    success, result, error = hostile_test(categorize_function, "hostile_func", fname)
    record_test("categorize_function", "hygiene.generate_docstrings", "hostile_names",
                success, error, "high" if "DROP" in str(fname) else "medium")

# Test generate_docstring_template
templates = [
    {"func_name": "normal", "params": [], "return_type": None},
    {"func_name": "'; DROP TABLE docs; --", "params": [], "return_type": "str"},
    {"func_name": "<script>alert(1)</script>", "params": [], "return_type": "Any"},
    {"func_name": "../../../etc/passwd", "params": [], "return_type": "str"},
    {"func_name": "test", "params": [{"name": "'; DROP", "annotation": "str"}], "return_type": "str"},
]

for inp in templates:
    success, result, error = hostile_test(generate_docstring_template, "hostile_template", **inp)
    record_test("generate_docstring_template", "hygiene.generate_docstrings", "hostile_templates",
                success, error, "high" if "DROP" in str(inp.get('func_name','')) else "medium")

# Test find_python_files
hostile_paths = [
    Path("../../../etc"), Path("/dev/null"), Path("/nonexistent"),
    Path("path; rm -rf /"), Path("A" * 1000)
]

for path in hostile_paths:
    success, result, error = hostile_test(find_python_files, "hostile_path", path, False)
    record_test("find_python_files", "hygiene.generate_docstrings", "hostile_paths",
                success, error, "high" if "../" in str(path) else "medium")

# Test process_python_file
for path in hostile_paths:
    success, result, error = hostile_test(process_python_file, "hostile_process", path)
    record_test("process_python_file", "hygiene.generate_docstrings", "hostile_paths",
                success, error, "high" if "../" in str(path) else "medium")

print(f"Completed {len([r for r in test_results['module'] if 'hygiene' in r])} hygiene tests")

Completed 25 hygiene tests


## Section 3: Development Module - Architecture Analysis

In [4]:
from siege_utilities import (
    analyze_package_structure, generate_architecture_diagram
)

# Test analyze_package_structure
hostile_packages = [
    "", "../../../etc/passwd", "'; DROP TABLE packages; --",
    "<script>alert(1)</script>", "A" * 1000, "package\x00name",
    "nonexistent_package", "os", "sys"
]

for pkg in hostile_packages:
    success, result, error = hostile_test(analyze_package_structure, "hostile_pkg", pkg)
    record_test("analyze_package_structure", "development.architecture", "hostile_packages",
                success, error, "critical" if "DROP" in str(pkg) else "high" if "../" in str(pkg) else "medium")

# Test generate_architecture_diagram
configs = [
    {"output_format": "text", "output_file": None},
    {"output_format": "json", "output_file": "../../../etc/passwd"},
    {"output_format": "markdown", "output_file": "/dev/null"},
    {"output_format": "'; DROP TABLE files; --", "output_file": "test.txt"},
]

for cfg in configs:
    success, result, error = hostile_test(generate_architecture_diagram, "hostile_diagram", **cfg)
    record_test("generate_architecture_diagram", "development.architecture", "hostile_configs",
                success, error, "high" if "../" in str(cfg.get('output_file','')) else "medium")

print(f"Completed {len([r for r in test_results['module'] if 'development' in r])} development tests")

Architecture diagram saved to: ../../../etc/passwd
Architecture diagram saved to: /dev/null
Completed 13 development tests


## Section 4: Generate Phase 3 Results

In [5]:
df = pd.DataFrame(test_results)

print("\n" + "="*80)
print("PHASE 3 HOSTILE TESTING SUMMARY")
print("="*80)
print(f"\nTests run: {len(df)}")
print(f"Passed: {df['passed'].sum()}")
print(f"Failed: {(~df['passed']).sum()}")
print(f"Pass rate: {df['passed'].sum() / len(df) * 100:.1f}%")

unique = df['function'].nunique()
print(f"\nUnique functions: {unique}")
print(f"Phase 3 coverage: {unique}/751 = {unique/751*100:.1f}%")

print("\n" + "="*80)
print("RESULTS BY MODULE")
print("="*80)
summary = df.groupby('module').agg({'passed': ['sum', 'count']})
summary.columns = ['passed', 'total']
summary['pass_rate'] = (summary['passed'] / summary['total'] * 100).round(1)
print(summary)

failures = df[~df['passed']]
if len(failures) > 0:
    print("\n" + "="*80)
    print("FAILURES BY SEVERITY")
    print("="*80)
    print(failures.groupby('severity').size())
    
    critical_high = failures[failures['severity'].isin(['critical', 'high'])]
    if len(critical_high) > 0:
        print("\n" + "="*80)
        print("CRITICAL/HIGH FAILURES (First 5)")
        print("="*80)
        for idx, row in critical_high.head(5).iterrows():
            print(f"\n{row['function']} ({row['severity']})")
            print(f"  Module: {row['module']}")
            print(f"  Error: {row['error_message'][:100]}")
else:
    print("\n✅ NO FAILURES")

df.to_csv("hostile_testing_phase3_results.csv", index=False)
print(f"\nSaved: hostile_testing_phase3_results.csv")

print("\n" + "="*80)
print("FUNCTIONS TESTED")
print("="*80)
for func in df['function'].unique():
    tests = df[df['function'] == func]
    passed = tests['passed'].sum()
    total = len(tests)
    print(f"{func}: {passed}/{total} ({passed/total*100:.0f}%)")

print("\n" + "="*80)
print("CUMULATIVE COVERAGE (Phases 1-3)")
print("="*80)
total_unique = 11 + unique
print(f"Total unique functions: ~{total_unique}")
print(f"Cumulative coverage: ~{total_unique}/751 = {total_unique/751*100:.1f}%")


PHASE 3 HOSTILE TESTING SUMMARY

Tests run: 48
Passed: 32
Failed: 16
Pass rate: 66.7%

Unique functions: 8
Phase 3 coverage: 8/751 = 1.1%

RESULTS BY MODULE
                             passed  total  pass_rate
module                                               
data.sample_data                 10     10      100.0
development.architecture         13     13      100.0
hygiene.generate_docstrings       9     25       36.0

FAILURES BY SEVERITY
severity
high       3
medium    13
dtype: int64

CRITICAL/HIGH FAILURES (First 5)

generate_docstring_template (high)
  Module: hygiene.generate_docstrings
  Error: generate_docstring_template() got an unexpected keyword argument 'params'

find_python_files (high)
  Module: hygiene.generate_docstrings
  Error: find_python_files() takes 1 positional argument but 2 were given

process_python_file (high)
  Module: hygiene.generate_docstrings
  Error: '../../../etc' is not in the subpath of '/Users/dheerajchand/Desktop/claude/siege_utilities_projec


Saved: hostile_testing_phase3_results.csv

FUNCTIONS TESTED
list_available_datasets: 1/1 (100%)
get_dataset_info: 9/9 (100%)
categorize_function: 9/10 (90%)
generate_docstring_template: 0/5 (0%)
find_python_files: 0/5 (0%)
process_python_file: 0/5 (0%)
analyze_package_structure: 9/9 (100%)
generate_architecture_diagram: 4/4 (100%)

CUMULATIVE COVERAGE (Phases 1-3)
Total unique functions: ~19
Cumulative coverage: ~19/751 = 2.5%
